# Notebook 03 — Reaction-Diffusion Systems

**Module:** 15 — Simulation and Agent-Based Modeling  
**Tier:** 2 — Working competence  
**Notebook:** 3 of 12  
**Time estimate:** 80 minutes

> How does a leopard get its spots? How does a zebrafish get its stripes?
> In 1952, Alan Turing proposed that spatial patterns in biology arise from
> the interaction of a fast-diffusing inhibitor and a slow-diffusing activator.
> This notebook implements the theory computationally using finite differences.

---
## Step 1 — Motivation

A reaction-diffusion (RD) system combines local chemical reactions with spatial
diffusion. Remarkably, even when the homogeneous steady state is stable (a flat,
spatially uniform concentration), diffusion can *destabilize* it — a phenomenon
called **diffusion-driven instability** or the **Turing instability**. The result
is spontaneous spatial pattern formation: spots, stripes, spirals — without any
pre-existing spatial template. This is how morphogenesis works in developing embryos.

---
## Step 2 — Intuition

**Activator-inhibitor intuition:**
- Activator (u): promotes itself *and* the inhibitor; diffuses slowly
- Inhibitor (v): suppresses the activator; diffuses fast

Imagine a patch of activator forming. It excites nearby inhibitor strongly
(fast diffusion), which spreads and suppresses the activator everywhere except
the original patch. Meanwhile, the activator builds up in its local patch.
Result: a stable high-activator spot surrounded by low-activator inhibited territory.

**Gray-Scott model:** a specific, well-studied RD system with two species U and V:
- Feed: fresh U enters at rate f
- Reaction: U + 2V → 3V (autocatalytic — one V catalyzes conversion of U to V)
- Kill: V removed at rate k
- Diffusion: D_u >> D_v (or D_u = 2D_v in some parametrisations)
- Depending on (f, k), produces spots, stripes, worms, chaos

**Finite differences:** approximate the Laplacian $\nabla^2 u$ on a grid:
$$\nabla^2 u_{i,j} \approx \frac{u_{i+1,j} + u_{i-1,j} + u_{i,j+1} + u_{i,j-1} - 4u_{i,j}}{\Delta x^2}$$

---
## Step 3 — Biological Background

**Evidence for Turing patterns:**
- Zebrafish skin: pigment cells (melanophores, xanthophores) interact as
  activator-inhibitor; genetic perturbations change stripe to spot pattern
  as predicted by Turing theory (Nakamasu et al. 2009, PNAS).
- Mouse palate rugae: knockout of SPRY2 (inhibitor gene) changes the periodic
  ridge pattern — consistent with Turing instability.
- Digit formation: reaction-diffusion models reproduce the spacing of fingers
  and the polydactyly phenotype in some mutants.
- Bacterial colonies: *E. coli* under nutrient limitation form spatial rings
  through a chemotaxis-based RD mechanism.

**Caveats:** the Turing mechanism is not universal — some patterns arise from
mechanical tension, oriented growth, or morphogen gradients. Demonstrating
Turing in a real system requires showing both the activator-inhibitor pair
and the diffusion ratio D_v/D_u >> 1.

---
## Step 4 — Mathematical Explanation

**General RD system:**
$$\frac{\partial u}{\partial t} = D_u \nabla^2 u + f(u, v)$$
$$\frac{\partial v}{\partial t} = D_v \nabla^2 v + g(u, v)$$

**Gray-Scott model:**
$$f(u, v) = -uv^2 + F(1-u)$$
$$g(u, v) = uv^2 - (F+k)v$$

where $F$ (feed rate) and $k$ (kill rate) are the control parameters.
The autocatalytic term $uv^2$ means: one molecule of U and two of V produce three V.

**Turing instability condition:** the homogeneous steady state $(u^*, v^*)$ is
stable without diffusion but unstable with diffusion for some wavenumber $q^2 > 0$.
This requires:
$$f_u + g_v < 0 \text{ (trace condition — stability without diffusion)}$$
$$D_v f_u + D_u g_v > 2\sqrt{D_u D_v (f_u g_v - f_v g_u)} \text{ (Turing condition)}$$
Note: for the second condition to hold with $D_u < D_v$, we need
$f_u > 0$ (activator self-activates) and $g_v < 0$ (inhibitor self-inhibits).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation
from matplotlib.colors import Normalize

rng = np.random.default_rng(42)

# ---- Gray-Scott simulation ----
def gray_scott_step(U, V, Du, Dv, F, k, dt, dx):
    """
    One Euler time step of the Gray-Scott reaction-diffusion system.
    Uses finite differences with periodic boundary conditions (np.roll).
    """
    lap_U = (
        np.roll(U, 1, axis=0) + np.roll(U, -1, axis=0)
      + np.roll(U, 1, axis=1) + np.roll(U, -1, axis=1)
      - 4 * U
    ) / dx**2
    lap_V = (
        np.roll(V, 1, axis=0) + np.roll(V, -1, axis=0)
      + np.roll(V, 1, axis=1) + np.roll(V, -1, axis=1)
      - 4 * V
    ) / dx**2
    # Reaction terms
    react = U * V**2
    dU = Du * lap_U - react + F * (1 - U)
    dV = Dv * lap_V + react - (F + k) * V
    return U + dt * dU, V + dt * dV

def run_gray_scott(N=100, n_steps=5000, Du=0.16, Dv=0.08, F=0.060, k=0.062,
                    dt=1.0, dx=1.0, seed=42):
    """
    Run Gray-Scott simulation.
    Default parameters produce spot pattern (coral growth).
    Returns final U, V grids.
    """
    rng_local = np.random.default_rng(seed)
    # Initial conditions: all U=1, all V=0, with a small random perturbation
    U = np.ones((N, N))
    V = np.zeros((N, N))
    # Seed a square of V in the centre
    cx, cy = N//2, N//2
    sq = 5
    U[cx-sq:cx+sq, cy-sq:cy+sq] = 0.5
    V[cx-sq:cx+sq, cy-sq:cy+sq] = 0.25
    # Add tiny noise to break symmetry
    U += 0.01 * rng_local.standard_normal((N, N))
    V += 0.01 * rng_local.standard_normal((N, N))
    U = np.clip(U, 0, 1); V = np.clip(V, 0, 1)
    
    for _ in range(n_steps):
        U, V = gray_scott_step(U, V, Du, Dv, F, k, dt, dx)
    return U, V

# Run three parameter regimes
print('Running Gray-Scott (spots)...')
U_spots, V_spots = run_gray_scott(F=0.060, k=0.062, n_steps=8000)
print('Running Gray-Scott (stripes)...')
U_stripes, V_stripes = run_gray_scott(F=0.040, k=0.060, n_steps=8000)
print('Running Gray-Scott (worms)...')
U_worms, V_worms = run_gray_scott(F=0.055, k=0.065, n_steps=8000)
print('Done.')

In [ ]:
# ---- Turing instability analysis (linearized) ----
# Using a simple activator-inhibitor (Gierer-Meinhardt-like)
def turing_dispersion(q2_vals, fu, fv, gu, gv, Du, Dv):
    """
    Growth rate of wavenumber q^2 under linearization around homogeneous SS.
    Turing unstable if max(sigma(q^2)) > 0 for some q^2 > 0.
    """
    sigmas = []
    for q2 in q2_vals:
        # 2x2 Jacobian with diffusion: A - D*q^2 I
        # characteristic polynomial -> eigenvalues
        a11 = fu - Du * q2
        a12 = fv
        a22 = gv - Dv * q2
        a21 = gu
        trace = a11 + a22
        det   = a11*a22 - a12*a21
        disc  = trace**2 - 4*det
        lam1  = (trace + np.sqrt(disc + 0j)) / 2
        lam2  = (trace - np.sqrt(disc + 0j)) / 2
        sigmas.append(max(lam1.real, lam2.real))
    return np.array(sigmas)

# Example activator-inhibitor Jacobian at SS
# (stable without diffusion: trace < 0, det > 0)
fu, fv = 3.0, -5.0  # activator: self-activates, inhibited by v
gu, gv = 2.0, -3.0  # inhibitor: produced by u, self-regulates
Du_ai, Dv_ai = 1.0, 10.0  # inhibitor diffuses 10x faster

q2_range = np.linspace(0, 3.0, 300)
sigma_ai = turing_dispersion(q2_range, fu, fv, gu, gv, Du_ai, Dv_ai)
q2_peak = q2_range[np.argmax(sigma_ai)]
print(f'Turing dispersion: peak sigma at q^2={q2_peak:.3f} (wavenumber q={np.sqrt(q2_peak):.3f})')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Panel A: spots
ax = axes[0, 0]
im = ax.imshow(V_spots, cmap='hot', origin='lower', interpolation='bilinear')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('A. Gray-Scott: spots\n(F=0.060, k=0.062)')
ax.axis('off')

# Panel B: stripes
ax = axes[0, 1]
im = ax.imshow(V_stripes, cmap='hot', origin='lower', interpolation='bilinear')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('B. Gray-Scott: stripes\n(F=0.040, k=0.060)')
ax.axis('off')

# Panel C: worms
ax = axes[0, 2]
im = ax.imshow(V_worms, cmap='hot', origin='lower', interpolation='bilinear')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('C. Gray-Scott: worms\n(F=0.055, k=0.065)')
ax.axis('off')

# Panel D: Turing dispersion relation
ax = axes[1, 0]
ax.plot(q2_range, sigma_ai, 'steelblue', lw=2)
ax.axhline(0, color='k', lw=0.8)
ax.axvline(q2_peak, color='tomato', ls='--', lw=1.5, label=f'Peak at q²={q2_peak:.2f}')
ax.fill_between(q2_range, sigma_ai, 0,
                  where=(sigma_ai > 0), alpha=0.2, color='tomato', label='Turing unstable')
ax.set_xlabel('Wavenumber² (q²)')
ax.set_ylabel('Growth rate σ(q²)')
ax.set_title('D. Turing dispersion relation\n(activator-inhibitor system)')
ax.legend(fontsize=8)

# Panel E: Gray-Scott (f,k) parameter space sketch
ax = axes[1, 1]
# Approximate regime boundaries (from Pearson 1993 classification)
f_vals = np.linspace(0.01, 0.10, 200)
ax.fill_between(f_vals, 0.040, 0.052, alpha=0.3, color='steelblue', label='Stripes')
ax.fill_between(f_vals, 0.056, 0.067, alpha=0.3, color='tomato', label='Spots')
ax.fill_between(f_vals, 0.052, 0.056, alpha=0.3, color='green', label='Worms')
ax.plot(0.060, 0.062, 'ro', ms=8, label='Spot example (A)')
ax.plot(0.040, 0.060, 'b^', ms=8, label='Stripe example (B)')
ax.plot(0.055, 0.065, 'gs', ms=8, label='Worm example (C)')
ax.set_xlabel('Feed rate F'); ax.set_ylabel('Kill rate k')
ax.set_title('E. Gray-Scott parameter space\n(approximate pattern regimes)')
ax.legend(fontsize=7)

# Panel F: 1D reaction-diffusion profile for spots
ax = axes[1, 2]
row = V_spots[50, :]
ax.plot(row, 'steelblue', lw=2, label='V concentration')
row_u = U_spots[50, :]
ax.plot(row_u, 'tomato', lw=2, label='U concentration')
ax.set_xlabel('Grid position (x)'); ax.set_ylabel('Concentration')
ax.set_title('F. 1D cross-section of spot pattern\n(complementary U/V concentrations)')
ax.legend(fontsize=9)

plt.suptitle('Module 15 NB03: Reaction-Diffusion Systems', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('reaction_diffusion_systems.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Move through the Gray-Scott (F, k) parameter space systematically: sweep F
   from 0.02 to 0.08 at fixed k=0.062, running 4000 steps each. Show the final
   V grid for 5 values of F. Document the pattern transition.
2. Show that the Turing instability requires D_v > D_u (diffusion asymmetry).
   Plot the dispersion relation σ(q²) for D_v/D_u = 1, 2, 5, 10 and show
   at which ratio the instability first appears.
3. Implement a simple Gierer-Meinhardt model (a different activator-inhibitor
   kinetics) and show it produces similar Turing patterns to Gray-Scott.

---
## Step 10 — Quiz

1. What is the Turing instability? Why is it counterintuitive that diffusion
   can *create* spatial structure rather than destroying it?
2. In the Gray-Scott model, what does the autocatalytic term uv² represent
   biologically (if U = activator morphogen, V = inhibitor morphogen)?
3. What numerical method is used to approximate the Laplacian in 2D? Write
   out the finite-difference formula for ∇²u at grid point (i,j).
4. Why are periodic boundary conditions used in reaction-diffusion simulations?
   What are the alternatives and their trade-offs?

---
## Step 12 — Reflection

> *[In one paragraph: explain the Turing instability mechanism in plain language,
> starting from: "imagine a uniform sheet of activator and inhibitor..." Work
> through what happens when a small perturbation is introduced at one location,
> invoking the diffusion speed difference. Conclude with why this produces
> a periodic pattern rather than one big spot.]*

---
*Next: `04_cellular_automata.ipynb`*